# Claim 1 – mục 4: đo đặc trưng dữ liệu (không cần GPU)

Đo trên **tín hiệu mà RA nhận** (tiền xử lý giống pipeline, trước RA), theo từng subject, cho BCI IV-2a, BCI IV-2b, Zhou2016 và OpenBMI. Không train mô hình nào.

| Giả thuyết | Đo gì |
|---|---|
| **H1** tương quan giữa kênh / tham chiếu chung | \|r\| trung bình giữa các kênh; tỉ lệ năng lượng ở thành phần chung (phần CAR sẽ trừ đi) |
| **H2** năng lượng ngoài băng mu/beta | tỉ lệ năng lượng < 4 Hz, 4–8, 8–30, 30–45 Hz; độ biến thiên giữa subject. Zhou đo 2 lần: pipeline (8–30 Hz) và **không lọc** |
| **H3** chất lượng tham chiếu RA | độ lệch giữa tham chiếu tính từ hai nửa trial; số điều kiện; trial/kênh; khoảng cách tham chiếu session fit và session đánh giá |
| **H4** kiểu khác biệt giữa subject | trong tangent space, tách phương sai thành: giữa subject (RA khử được), giữa lớp, tương tác subject × lớp (RA không khử được), phần dư |

**Thiết lập Kaggle:** Accelerator = **None (CPU)**, Internet = **On**.

**Add Input:**
- `openbmi-dataset-cuong`: OpenBMI có CAR (bắt buộc nếu muốn có OpenBMI).
- `openbmi-nocar-cuong`: OpenBMI không CAR (tùy chọn, khi đã tạo xong).

Không gắn dataset nào thì notebook vẫn chạy cho 2a, 2b, Zhou.

Kết quả lưu ở `/kaggle/working/ra_data_characteristics/` và file zip cùng tên.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'moabb>=1.1'], check=True)
import os
os.environ.setdefault('MPLBACKEND', 'Agg')
import mne, moabb
print('mne', mne.__version__, '| moabb', moabb.__version__)


In [ ]:
%%writefile /tmp/ra_data_characteristics.py
"""Claim 1, item 4: measure the data characteristics that could make RA help BCI IV-2a most.

Everything is measured on the signals RA receives (pipeline preprocessing, before RA),
per subject, for BCI IV-2a, BCI IV-2b, Zhou2016 and OpenBMI. No model is trained.

H1  inter-channel correlation / common reference component
    mean |r| between channels; share of signal energy in the common (average) mode,
    i.e. the part that CAR would remove.
H2  out-of-band power (no band-pass on 2a/2b)
    share of 0.5-45 Hz power below 4 Hz, 4-8 Hz, 8-30 Hz and 30-45 Hz; between-subject
    spread of these shares. Zhou2016 is measured twice: pipeline (8-30 Hz) and unfiltered.
H3  quality of the RA reference
    split-half distance of the reference (fit sessions), its condition number, trials per
    channel, and the distance between the fit-session and evaluation-session references.
H4  kind of inter-subject variability
    tangent-space (log-Euclidean) sums of squares split into subject, class,
    subject x class and residual parts, before and after RA. RA removes the subject part;
    it cannot remove the subject x class part.

Used by ra_data_characteristics.ipynb (Kaggle, CPU only), which embeds this file.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.linalg import eigh
from scipy.integrate import trapezoid
from scipy.signal import welch
from sklearn.covariance import OAS

BANDS = {"low_0.5-4": (0.5, 4.0), "theta_4-8": (4.0, 8.0),
         "mubeta_8-30": (8.0, 30.0), "high_30-45": (30.0, 45.0)}
TOTAL_BAND = (0.5, 45.0)          # common to every dataset (OpenBMI is sampled at 100 Hz)
SPLIT_HALF_REPEATS = 10


# ---------------------------------------------------------------- SPD helpers
def spd_power(matrix: np.ndarray, exponent: float) -> np.ndarray:
    values, vectors = eigh(matrix)
    return (vectors * np.maximum(values, 1e-12) ** exponent) @ vectors.T


def spd_log(matrix: np.ndarray) -> np.ndarray:
    values, vectors = eigh(matrix)
    return (vectors * np.log(np.maximum(values, 1e-12))) @ vectors.T


def riemann_mean(covs: np.ndarray, tol: float = 1e-7, max_iter: int = 50) -> np.ndarray:
    """Affine-invariant Riemannian mean (same iteration as the RA used by the model)."""
    mean = covs.mean(axis=0)
    for _ in range(max_iter):
        root, invroot = spd_power(mean, 0.5), spd_power(mean, -0.5)
        tangent = np.mean([spd_log(invroot @ c @ invroot) for c in covs], axis=0)
        if np.linalg.norm(tangent, "fro") < tol:
            break
        values, vectors = eigh(tangent)
        mean = root @ ((vectors * np.exp(values)) @ vectors.T) @ root
        mean = (mean + mean.T) / 2
    return mean


def riemann_distance(a: np.ndarray, b: np.ndarray) -> float:
    values = eigh(b, a, eigvals_only=True)
    return float(np.linalg.norm(np.log(np.maximum(values, 1e-12))))


def oas_covariances(x: np.ndarray) -> np.ndarray:
    return np.stack([OAS().fit(trial.T.astype(np.float64)).covariance_ for trial in x])


def tangent_vectors(covs: np.ndarray) -> np.ndarray:
    """Log-Euclidean vectors: upper triangle of log(C), off-diagonal scaled by sqrt(2)."""
    n = covs.shape[1]
    rows, cols = np.triu_indices(n)
    weight = np.where(rows == cols, 1.0, np.sqrt(2.0))
    return np.stack([spd_log(c)[rows, cols] * weight for c in covs])


# ---------------------------------------------------------------- data container
@dataclass
class SubjectData:
    subject: int
    sessions: list            # [(x [trials, channels, samples] float32, y int), ...] chronological
    fit_sessions: int         # leading sessions used to fit the frozen RA reference


@dataclass
class View:
    name: str                 # e.g. "BCI IV-2a", "Zhou2016 (unfiltered)"
    sfreq: float
    preprocessing: str        # human-readable description
    subjects: list            # [SubjectData]


# ---------------------------------------------------------------- per-subject metrics
def h1_correlation(x: np.ndarray) -> dict:
    """Mean |r| between channels and the common-mode energy share, averaged over trials."""
    xc = x - x.mean(axis=-1, keepdims=True)
    cov = np.einsum("nct,ndt->ncd", xc, xc, optimize=True) / x.shape[-1]
    std = np.sqrt(np.einsum("ncc->nc", cov))
    corr = cov / (std[:, :, None] * std[:, None, :])
    n_ch = x.shape[1]
    off = ~np.eye(n_ch, dtype=bool)
    mean_abs_corr = float(np.abs(corr[:, off]).mean())
    # Energy of the channel average, as a share of total energy: what CAR would remove.
    common = cov.sum(axis=(1, 2)) / n_ch / np.einsum("ncc->n", cov)
    common_share = float(common.mean())
    return {
        "h1_mean_abs_corr": mean_abs_corr,
        "h1_common_mode_share": common_share,
        # 0 = independent equal-variance channels, 1 = identical channels.
        "h1_common_mode_excess": float((common_share - 1 / n_ch) / (1 - 1 / n_ch)),
    }


def mean_psd(x: np.ndarray, sfreq: float) -> tuple[np.ndarray, np.ndarray]:
    nperseg = int(min(x.shape[-1], 2 * sfreq))
    freqs, psd = welch(x.astype(np.float64), fs=sfreq, nperseg=nperseg, axis=-1)
    return freqs, psd.mean(axis=(0, 1))


def h2_band_shares(freqs: np.ndarray, psd: np.ndarray) -> dict:
    def power(lo, hi):
        keep = (freqs >= lo) & (freqs < hi)
        return float(trapezoid(psd[keep], freqs[keep])) if keep.sum() > 1 else 0.0

    total = power(*TOTAL_BAND)
    shares = {f"h2_share_{name}": power(lo, hi) / total for name, (lo, hi) in BANDS.items()}
    shares["h2_share_outside_8-30"] = 1.0 - shares["h2_share_mubeta_8-30"]
    return shares


def h3_reference(subject: SubjectData, rng: np.random.Generator) -> tuple[dict, list]:
    fit_x = np.concatenate([x for x, _ in subject.sessions[:subject.fit_sessions]])
    eval_x = np.concatenate([x for x, _ in subject.sessions[subject.fit_sessions:]])
    n_ch = fit_x.shape[1]
    fit_covs = oas_covariances(fit_x)
    eval_covs = oas_covariances(eval_x)
    reference = riemann_mean(fit_covs)
    halves = []
    for _ in range(SPLIT_HALF_REPEATS):
        order = rng.permutation(len(fit_covs))
        a, b = order[: len(order) // 2], order[len(order) // 2:]
        halves.append(riemann_distance(riemann_mean(fit_covs[a]), riemann_mean(fit_covs[b])))
    values = eigh(reference, eigvals_only=True)
    metrics = {
        "h3_fit_trials": int(len(fit_covs)),
        "h3_trials_per_channel": float(len(fit_covs) / n_ch),
        "h3_split_half_distance": float(np.mean(halves) / np.sqrt(n_ch)),
        "h3_reference_log10_condition": float(np.log10(values.max() / max(values.min(), 1e-12))),
        "h3_session_shift": float(riemann_distance(reference, riemann_mean(eval_covs)) / np.sqrt(n_ch)),
    }
    return metrics, [fit_covs, eval_covs, reference]


# ---------------------------------------------------------------- H4: variance decomposition
def variance_decomposition(vectors: np.ndarray, subjects: np.ndarray, labels: np.ndarray) -> tuple[dict, pd.DataFrame]:
    """Sums of squares of tangent vectors split into subject, class, subject x class, residual."""
    grand = vectors.mean(axis=0)
    total = float(((vectors - grand) ** 2).sum())

    def between(groups):
        ss = 0.0
        for g in np.unique(groups, axis=0):
            mask = np.all(groups == g, axis=1) if groups.ndim == 2 else groups == g
            ss += mask.sum() * float(((vectors[mask].mean(axis=0) - grand) ** 2).sum())
        return ss

    ss_subject = between(subjects)
    ss_class = between(labels)
    ss_cell = between(np.stack([subjects, labels], axis=1))
    ss_interaction = max(ss_cell - ss_subject - ss_class, 0.0)
    summary = {
        "eta2_subject": ss_subject / total,
        "eta2_class": ss_class / total,
        "eta2_subject_x_class": ss_interaction / total,
        "eta2_residual": (total - ss_cell) / total,
    }
    summary["subject_over_interaction"] = summary["eta2_subject"] / max(summary["eta2_subject_x_class"], 1e-12)
    summary["class_over_interaction"] = summary["eta2_class"] / max(summary["eta2_subject_x_class"], 1e-12)

    # Per subject: offset of its mean (removable by RA) vs its class-specific deviation (not removable).
    p = vectors.shape[1]
    class_means = {k: vectors[labels == k].mean(axis=0) for k in np.unique(labels)}
    rows = []
    for s in np.unique(subjects):
        ms = subjects == s
        subject_mean = vectors[ms].mean(axis=0)
        offset = float(np.linalg.norm(subject_mean - grand) / np.sqrt(p))
        dev = [np.sum((vectors[ms & (labels == k)].mean(axis=0) - subject_mean - class_means[k] + grand) ** 2)
               for k in class_means if np.any(ms & (labels == k))]
        interaction = float(np.sqrt(np.mean(dev) / p))
        rows.append({"subject": int(s), "h4_subject_offset": offset,
                     "h4_subject_x_class": interaction,
                     "h4_offset_over_interaction": offset / max(interaction, 1e-12)})
    return summary, pd.DataFrame(rows)


# ---------------------------------------------------------------- one view
def analyse_view(view: View, seed: int = 0, log=print) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(seed)
    rows, psd_rows = [], []
    vec_raw, vec_shape, vec_ra, subj_ids, labels = [], [], [], [], []
    for sd in view.subjects:
        x_all = np.concatenate([x for x, _ in sd.sessions])
        y_all = np.concatenate([y for _, y in sd.sessions])
        row = {"view": view.name, "subject": sd.subject, "n_channels": x_all.shape[1],
               "n_classes": int(len(np.unique(y_all))), "n_trials": int(len(y_all))}
        row.update(h1_correlation(x_all))
        freqs, psd = mean_psd(x_all, view.sfreq)
        row.update(h2_band_shares(freqs, psd))
        keep = (freqs >= TOTAL_BAND[0]) & (freqs <= TOTAL_BAND[1])
        total = trapezoid(psd[keep], freqs[keep])
        psd_rows += [{"view": view.name, "subject": sd.subject, "freq": float(f), "relative_psd": float(v / total)}
                     for f, v in zip(freqs[keep], psd[keep])]
        h3, (fit_covs, eval_covs, reference) = h3_reference(sd, rng)
        row.update(h3)
        rows.append(row)

        covs = np.concatenate([fit_covs, eval_covs])
        whiten = spd_power(reference, -0.5)
        traces = np.einsum("ncc->n", covs)
        vec_raw.append(tangent_vectors(covs))
        vec_shape.append(tangent_vectors(covs / traces[:, None, None] * covs.shape[1]))
        vec_ra.append(tangent_vectors(np.einsum("ij,njk,kl->nil", whiten, covs, whiten, optimize=True)))
        fit_y = np.concatenate([y for _, y in sd.sessions[:sd.fit_sessions]])
        eval_y = np.concatenate([y for _, y in sd.sessions[sd.fit_sessions:]])
        labels.append(np.concatenate([fit_y, eval_y]))
        subj_ids.append(np.full(len(covs), sd.subject))
        log(f"  {view.name} S{sd.subject:02d}: corr={row['h1_mean_abs_corr']:.2f} "
            f"outside8-30={row['h2_share_outside_8-30']:.2f} split-half={row['h3_split_half_distance']:.3f}")

    subjects = np.concatenate(subj_ids)
    labels = np.concatenate(labels)
    decomposition, per_subject = [], None
    for kind, vectors in (("before RA", vec_raw), ("before RA, trace-normalized", vec_shape),
                          ("after frozen RA", vec_ra)):
        summary, table = variance_decomposition(np.concatenate(vectors), subjects, labels)
        decomposition.append({"view": view.name, "covariance": kind, **summary})
        if kind == "before RA":
            per_subject = table
    metrics = pd.DataFrame(rows).merge(per_subject, on="subject")
    return metrics, pd.DataFrame(decomposition), pd.DataFrame(psd_rows)


# ---------------------------------------------------------------- summaries
SUMMARY_METRICS = [
    "h1_mean_abs_corr", "h1_common_mode_share", "h1_common_mode_excess",
    "h2_share_low_0.5-4", "h2_share_theta_4-8", "h2_share_mubeta_8-30", "h2_share_high_30-45",
    "h2_share_outside_8-30",
    "h3_trials_per_channel", "h3_split_half_distance", "h3_reference_log10_condition", "h3_session_shift",
    "h4_subject_offset", "h4_subject_x_class", "h4_offset_over_interaction",
]


def summarise(metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for view, group in metrics.groupby("view", sort=False):
        row = {"view": view, "n_subjects": len(group), "n_channels": int(group["n_channels"].iloc[0])}
        for metric in SUMMARY_METRICS:
            q1, med, q3 = group[metric].quantile([0.25, 0.5, 0.75])
            row[metric] = f"{med:.3f} [{q1:.3f}-{q3:.3f}]"
        # Between-subject spread of the H1/H2 shares (coefficient of variation).
        for metric in ("h1_common_mode_share", "h2_share_outside_8-30", "h2_share_low_0.5-4"):
            values = group[metric]
            row[f"{metric}_between_subject_cv"] = f"{values.std(ddof=0) / values.mean():.3f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index("view").T


def natural_key(value) -> tuple:
    return tuple(int(part) if part.isdigit() else part for part in re.split(r"(\d+)", str(value)))


def save_json(path: Path, payload: dict) -> None:
    path.write_text(json.dumps(payload, indent=2, default=float))


In [ ]:
import json, shutil, sys, time
from pathlib import Path
import numpy as np
import pandas as pd
import mne
from moabb import datasets as moabb_datasets
from moabb.utils import set_download_dir

sys.path.insert(0, '/tmp')
import ra_data_characteristics as rdc

mne.set_log_level('ERROR')
DATA_DIR = Path('/tmp/mne_data'); DATA_DIR.mkdir(parents=True, exist_ok=True)
set_download_dir(str(DATA_DIR))
OUT = Path('/kaggle/working/ra_data_characteristics'); OUT.mkdir(parents=True, exist_ok=True)

# Same windows and filters as the training pipeline (configs/hada_tcformer.yaml):
# BCI IV-2a full MOABB interval, no filter; BCI IV-2b interval minus its last 0.5 s, no filter;
# Zhou2016 8-30 Hz and 0.5-5.0 s after the cue. fit = leading sessions used for the frozen RA reference.
SPECS = {
    'BCI IV-2a': dict(classes=('BNCI2014_001', 'BNCI2014001'), start=0.0, duration=4.0, fit=1, sessions=2),
    'BCI IV-2b': dict(classes=('BNCI2014_004', 'BNCI2014004'), start=0.0, duration=4.0, fit=3, sessions=5),
    'Zhou2016': dict(classes=('Zhou2016',), start=0.5, duration=4.5, fit=2, sessions=3),
}


def moabb_dataset(names):
    for name in names:
        if hasattr(moabb_datasets, name):
            return getattr(moabb_datasets, name)()
    raise ImportError(f'none of {names} in this MOABB version')


def epochs_from_raw(raw, dataset, start, duration, band):
    event_id = dataset.event_id
    stim = [ch for ch, kind in zip(raw.ch_names, raw.get_channel_types()) if kind == 'stim']
    if stim:
        events = mne.find_events(raw, stim_channel=stim[0], shortest_event=1, verbose=False)
    else:
        events, _ = mne.events_from_annotations(raw, event_id=event_id, verbose=False)
    codes = sorted(event_id.values())
    events = events[np.isin(events[:, 2], codes)]
    eeg = raw.copy().pick('eeg').load_data(verbose=False)
    eeg.apply_function(lambda values: values * 1e6, channel_wise=False)   # V -> uV, as the pipeline
    if band is not None:
        eeg.filter(band[0], band[1], verbose=False)
    sfreq = eeg.info['sfreq']
    tmin = float(dataset.interval[0]) + start
    present = {name: code for name, code in event_id.items() if code in set(events[:, 2])}
    epochs = mne.Epochs(eeg, events, event_id=present, tmin=tmin, tmax=tmin + duration - 1.0 / sfreq,
                        baseline=None, preload=True, verbose=False)
    y = np.searchsorted(codes, epochs.events[:, 2]).astype(np.int64)
    return epochs.get_data().astype(np.float32), y, sfreq


def load_moabb_views(name, spec, bands):
    """One View per band setting; raw files are read once per subject."""
    dataset = moabb_dataset(spec['classes'])
    subjects = {label: [] for label in bands}
    sfreq = None
    for subject in dataset.subject_list:
        started = time.time()
        data = dataset.get_data(subjects=[subject])[subject]
        session_names = sorted(data, key=rdc.natural_key)
        if len(session_names) != spec['sessions']:
            raise RuntimeError(f'{name} S{subject}: {len(session_names)} sessions, expected {spec["sessions"]}')
        for label, band in bands.items():
            sessions = []
            for session in session_names:
                parts = [epochs_from_raw(data[session][run], dataset, spec['start'], spec['duration'], band)
                         for run in sorted(data[session], key=rdc.natural_key)]
                sfreq = parts[0][2]
                sessions.append((np.concatenate([p[0] for p in parts]), np.concatenate([p[1] for p in parts])))
            subjects[label].append(rdc.SubjectData(subject, sessions, spec['fit']))
        shapes = [s[0].shape for s in subjects[next(iter(bands))][-1].sessions]
        print(f'{name} S{subject:02d}: sessions {shapes} ({time.time() - started:.0f} s)', flush=True)
    return [rdc.View(label, sfreq, desc, subjects[label]) for label, desc in
            ((label, f'{spec["start"]}-{spec["start"] + spec["duration"]} s after interval start, band {band}')
             for label, band in bands.items())]


def load_openbmi_views():
    """OpenBMI caches attached under /kaggle/input, grouped by their preprocessing signature."""
    groups = {}
    for path in sorted(Path('/kaggle/input').rglob('openbmi_S*.npz')):
        with np.load(path, allow_pickle=False) as data:
            signature = json.loads(str(data['signature']))
        car = 'CAR' if signature.get('common_average_reference', True) else 'no CAR'
        groups.setdefault((car, signature['sfreq'], signature['low_cut'], signature['high_cut']), {}) \
              .setdefault(int(path.stem[-2:]), path)
    views = []
    for (car, sfreq, low, high), files in sorted(groups.items()):
        if len(files) != 54:
            print(f'OpenBMI {car}: only {len(files)}/54 subjects attached, used as is', flush=True)
        subjects = []
        for subject in sorted(files):
            with np.load(files[subject], allow_pickle=False) as data:
                subjects.append(rdc.SubjectData(subject, [(data['X1'], data['y1']), (data['X2'], data['y2'])], 1))
        views.append(rdc.View(f'OpenBMI ({car}, {low:g}-{high:g} Hz)', float(sfreq),
                              f'{car}, {low}-{high} Hz, {sfreq} Hz, 0-4 s', subjects))
        print(f'OpenBMI {car}: {len(subjects)} subjects', flush=True)
    if not views:
        print('No OpenBMI cache attached: add openbmi-dataset-cuong (and openbmi-nocar-cuong) via Add Input.')
    return views


In [ ]:
views = []
views += load_moabb_views('BCI IV-2a', SPECS['BCI IV-2a'], {'BCI IV-2a': None})
views += load_moabb_views('BCI IV-2b', SPECS['BCI IV-2b'], {'BCI IV-2b': None})
views += load_moabb_views('Zhou2016', SPECS['Zhou2016'],
                          {'Zhou2016 (pipeline, 8-30 Hz)': (8.0, 30.0), 'Zhou2016 (unfiltered)': None})
views += load_openbmi_views()

metrics, decomposition, psd = [], [], []
for view in views:
    started = time.time()
    print(f'== {view.name}: {len(view.subjects)} subjects, {view.sfreq:g} Hz', flush=True)
    m, d, p = rdc.analyse_view(view)
    metrics.append(m); decomposition.append(d); psd.append(p)
    print(f'   done in {(time.time() - started) / 60:.1f} min', flush=True)
metrics = pd.concat(metrics, ignore_index=True)
decomposition = pd.concat(decomposition, ignore_index=True)
psd = pd.concat(psd, ignore_index=True)
metrics.to_csv(OUT / 'subject_metrics.csv', index=False)
decomposition.to_csv(OUT / 'variance_decomposition.csv', index=False)
psd.to_csv(OUT / 'relative_psd.csv', index=False)


In [ ]:
# RA gain of the network (tab:ablation, fixed RA / per-session RA) for reference.
RA_GAIN = {'BCI IV-2a': '+5.83 / +7.45', 'BCI IV-2b': '+0.51 / +0.52',
           'Zhou2016 (pipeline, 8-30 Hz)': '+0.33 / +1.33', 'Zhou2016 (unfiltered)': 'not run',
           'OpenBMI (CAR, 4-38 Hz)': '-2.89 / +0.50', 'OpenBMI (no CAR, 4-38 Hz)': 'running'}
summary = rdc.summarise(metrics)
summary.loc['network RA gain (fixed / per-session)'] = [RA_GAIN.get(v, '') for v in summary.columns]
summary.to_csv(OUT / 'summary.csv')
pd.set_option('display.max_colwidth', 40, 'display.width', 250)
display(summary)
display(decomposition.round(3))


In [ ]:
import matplotlib.pyplot as plt
order = list(dict.fromkeys(metrics['view']))

# H2: median relative power spectrum per view.
fig, ax = plt.subplots(figsize=(8, 4.5))
for view in order:
    curve = psd[psd['view'] == view].groupby('freq')['relative_psd'].median()
    ax.semilogy(curve.index, curve.values, label=view)
ax.axvspan(8, 30, color='0.9', zorder=0)
ax.set(xlabel='Frequency (Hz)', ylabel='Relative power (median over subjects)', title='H2: power spectrum seen by RA')
ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(OUT / 'h2_relative_psd.png', dpi=150); plt.show()

# H1-H4: per-subject distributions.
panels = [('h1_mean_abs_corr', 'H1: mean |r| between channels'),
          ('h1_common_mode_excess', 'H1: common-mode share (0 = none, 1 = all)'),
          ('h2_share_outside_8-30', 'H2: power share outside 8-30 Hz'),
          ('h3_split_half_distance', 'H3: split-half reference distance / sqrt(C)'),
          ('h3_session_shift', 'H3: fit vs evaluation reference / sqrt(C)'),
          ('h4_offset_over_interaction', 'H4: subject offset / subject x class')]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, (metric, title) in zip(axes.flat, panels):
    ax.boxplot([metrics.loc[metrics['view'] == v, metric] for v in order])
    ax.set_xticks(range(1, len(order) + 1), order, rotation=35, ha='right', fontsize=8)
    ax.set_title(title, fontsize=10)
fig.tight_layout(); fig.savefig(OUT / 'h1_h4_boxplots.png', dpi=150); plt.show()

# H4: variance decomposition before and after RA.
parts = ['eta2_subject', 'eta2_class', 'eta2_subject_x_class', 'eta2_residual']
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), sharey=True)
for ax, kind in zip(axes, ['before RA', 'after frozen RA']):
    table = decomposition[decomposition['covariance'] == kind].set_index('view').loc[order, parts]
    table.plot.bar(stacked=True, ax=ax, legend=ax is axes[0])
    ax.set_title(f'H4: tangent-space variance, {kind}'); ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=35)
fig.tight_layout(); fig.savefig(OUT / 'h4_variance_decomposition.png', dpi=150); plt.show()

shutil.make_archive(str(OUT), 'zip', root_dir=OUT)
print('Saved', sorted(p.name for p in OUT.iterdir()), 'and', f'{OUT}.zip')


## Cách đọc kết quả

| Nếu thấy | Thì |
|---|---|
| \|r\| và common-mode của **2a ≈ Zhou** (cùng đơn cực) nhưng Δ_RA rất khác | H1 **một mình không đủ** giải thích 2a và Zhou khác nhau |
| **2a** có tỉ lệ năng lượng ngoài 8–30 Hz cao và **biến thiên nhiều giữa subject**; **Zhou (unfiltered)** cũng cao nhưng **Zhou (pipeline)** thấp | Khác biệt 2a và Zhou đến từ **lựa chọn lọc tần số** (H2), không phải từ bản thân dữ liệu. Thí nghiệm tiếp theo: Zhou không lọc, có RA và không RA |
| OpenBMI có split-half distance cao nhất, trial/kênh thấp nhất | Tham chiếu RA của OpenBMI kém ổn định (H3) |
| 2a có `subject_over_interaction` (H4) cao nhất | Khác biệt giữa subject ở 2a chủ yếu là dịch chuyển chung, đúng loại RA sửa được |
| Sau RA, `eta2_subject` về ≈ 0 ở mọi bộ | Kiểm tra hợp lệ: RA khử đúng phần giữa subject |

Gửi lại `summary.csv`, `variance_decomposition.csv` và 3 hình (hoặc file zip) để phân tích tiếp.
